# 05 | Baseline Models - Sprint 3

## Objetivo del notebook

Entrenar y comparar los modelos baseline solicitados en Sprint 3. Estos modelos no son el final del proyecto; sirven como punto de comparación para justificar si el tuning y los ensambles realmente aportan.

Modelos incluidos:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine
5. K-Nearest Neighbors


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


## 1. Por qué se usa cada modelo

| Modelo | Por qué se incluye | Preprocesamiento necesario |
|---|---|---|
| Logistic Regression | Baseline lineal interpretable; útil para comparar contra modelos complejos. | Escalado + OHE + balanceo controlado. |
| Decision Tree | Modelo simple de reglas; interpretable pero propenso a overfitting. | OHE; no requiere escalado. |
| Random Forest | Reduce overfitting de un árbol individual; fuerte baseline tabular. | OHE; no requiere escalado; usa pesos de clase. |
| SVM | Buen clasificador para fronteras lineales/margen amplio; sensible a escala. | Escalado + OHE + balanceo controlado. |
| KNN | Baseline basado en distancia; sirve para detectar si casos similares predicen riesgo. | Escalado obligatorio + OHE + balanceo controlado. |

El objetivo no es que todos ganen, sino demostrar que se evaluaron familias distintas.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import RAW_DATA_PATH, TARGET, RANDOM_STATE
from src.preprocessing import split_X_y
from src.models import build_baseline_models
from src.evaluation import evaluate_model

df = pd.read_csv(RAW_DATA_PATH)
train, valid = train_test_split(df.sample(3000, random_state=RANDOM_STATE), test_size=0.30, stratify=df.sample(3000, random_state=RANDOM_STATE)[TARGET], random_state=RANDOM_STATE)
X_train, y_train = split_X_y(train)
X_valid, y_valid = split_X_y(valid)
models = build_baseline_models(X_train)
list(models.keys())


['LogisticRegression_baseline',
 'DecisionTree_baseline',
 'RandomForest_baseline',
 'SVM_baseline',
 'KNN_baseline']

## 2. Entrenamiento baseline

Cada modelo se entrena como pipeline completo. Esto significa que el modelo acepta columnas crudas y el pipeline hace internamente:

1. Feature engineering.
2. Imputación.
3. Encoding.
4. Escalado si corresponde.
5. Balanceo si corresponde.
6. Clasificación.

Esta estructura evita data leakage y es la misma que luego usa la API.


In [3]:
rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    metrics = evaluate_model(pipe, X_valid, y_valid)
    metrics.pop('classification_report', None)
    rows.append({'model': name, **metrics})

baseline_results = pd.DataFrame(rows).sort_values('business_value', ascending=False)
baseline_results[['model','business_value','recall','precision','f2','roc_auc','tp','fp','fn','tn']]


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


,model,business_value,recall,precision,f2,roc_auc,tp,fp,fn,tn
3,SVM_baseline,97800.0,0.205882,0.313433,0.221053,0.648496,21,46,81,752
2,RandomForest_baseline,89300.0,0.107843,0.687500,0.129717,0.670641,11,5,91,793
1,DecisionTree_baseline,88800.0,0.294118,0.241935,0.281955,0.588162,30,94,72,704
4,KNN_baseline,83300.0,0.196078,0.281690,0.208768,0.661482,20,51,82,747
0,LogisticRegression_baseline,38800.0,0.509804,0.184397,0.376812,0.675291,52,230,50,568


## 3. Cómo interpretar los resultados

- Si un modelo tiene recall alto pero precision muy baja, detecta Bad Buys pero bloquea demasiados autos buenos.
- Si tiene precision alta pero recall bajo, probablemente deja pasar Bad Buys; eso genera FN caros.
- La selección final no se hace aquí: estos resultados alimentan los candidatos de Sprint 4.
